# Lab 1 — Scalable AI Pipelines (MLOps Basics)

**Unit:** U24 · Phase F — ML Engineering / MLOps
**Goal:** Take a notebook model and add the production-grade scaffolding around it — pipelines, experiment tracking, model registry, serving, and drift monitoring.

In this lab you will:

1. Build a small **end-to-end ML pipeline** (data → features → train → evaluate).
2. Track experiments with **MLflow** (params, metrics, artifacts).
3. Save the model to a **model registry** (MLflow Model Registry).
4. Serve the model behind a **FastAPI** REST endpoint.
5. Detect simple **data drift** on incoming inputs.

> 📌 *We use libraries rather than building from scratch — this matches how MLOps is done in real teams.*


## 0. Setup — Install Libraries

We rely on `scikit-learn` (modeling), `mlflow` (tracking + registry), `fastapi` + `uvicorn` (serving), and `scipy` (drift test).


In [ ]:
# Install required libraries (run once)
# !pip install scikit-learn mlflow fastapi uvicorn scipy pandas numpy --quiet

# Import standard libraries
import os                                    # for file system operations
import numpy as np                           # numerical arrays
import pandas as pd                          # tabular data handling
import warnings                              # to silence noisy warnings
warnings.filterwarnings("ignore")            # cleaner output in the notebook
print("Libraries imported ✓")                # sanity check

## 1. Build an End-to-End Pipeline

> *Recall from the slides:* `Data → Features → Train → Evaluate → Deploy → Monitor`.

We will use **scikit-learn's `Pipeline`** so that feature transforms and the model are bundled into ONE object. This is the simplest defense against **training-serving skew** — there is only one code path.


In [ ]:
# Import sklearn components for the pipeline
from sklearn.datasets import load_breast_cancer             # toy classification dataset
from sklearn.model_selection import train_test_split        # split data into train/test
from sklearn.preprocessing import StandardScaler            # feature scaling transform
from sklearn.linear_model import LogisticRegression         # simple classifier
from sklearn.pipeline import Pipeline                       # bundles transforms + model
from sklearn.metrics import accuracy_score, f1_score        # evaluation metrics

# Load the dataset (binary classification: malignant vs benign)
data = load_breast_cancer(as_frame=True)                    # returns a Bunch with pandas frame
X = data.data                                               # feature columns
y = data.target                                             # binary labels

# Split into train and test (80/20) with a fixed seed for reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42                    # seed makes results reproducible
)
print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")  # check shapes

In [ ]:
# Define the pipeline: scaling step + classifier step
pipeline = Pipeline([
    ("scaler", StandardScaler()),                           # step 1: standardize features
    ("clf",    LogisticRegression(max_iter=1000))           # step 2: logistic regression
])

# Fit the entire pipeline on training data (both steps learn together)
pipeline.fit(X_train, y_train)                              # one call trains the whole flow

# Predict on test set using the SAME pipeline (no skew possible)
y_pred = pipeline.predict(X_test)                           # uses the fitted scaler + model

# Evaluate the predictions
acc = accuracy_score(y_test, y_pred)                        # overall accuracy
f1  = f1_score(y_test, y_pred)                              # harmonic mean of precision/recall
print(f"Accuracy: {acc:.4f}")                               # show accuracy
print(f"F1-score: {f1:.4f}")                                # show F1

✅ **Why this matters:** Because both the scaler *and* model live inside one Pipeline object, when we save it and reload it at serving time, **features are computed identically in training and production**.

## 2. Experiment Tracking with MLflow

> *Recall from the slides:* "Track every run: params, metrics, code version, data version, and artifacts."

MLflow logs each run to a local folder (`mlruns/`). We can later open the **MLflow UI** with `mlflow ui` to compare runs visually.


In [ ]:
import mlflow                                              # MLflow tracking library
import mlflow.sklearn                                      # sklearn-specific logging helpers

# Tell MLflow where to store experiment data (local folder)
mlflow.set_tracking_uri("file:./mlruns")                   # local file-based tracking server

# Create / select a named experiment (a logical group of runs)
mlflow.set_experiment("breast_cancer_classifier")          # auto-creates if not present
print("MLflow tracking set up ✓")                          # confirmation

In [ ]:
# Try a few different hyperparameter settings and log each as a run
C_values = [0.01, 0.1, 1.0, 10.0]                          # regularization strengths to test

# Loop through each candidate hyperparameter
for C in C_values:                                          # iterate over the list
    with mlflow.start_run(run_name=f"logreg_C={C}"):       # context = one tracked run

        # Build a fresh pipeline with this C value
        pipe = Pipeline([
            ("scaler", StandardScaler()),                  # same scaling step as before
            ("clf",    LogisticRegression(C=C, max_iter=1000))  # different regularization
        ])

        pipe.fit(X_train, y_train)                         # train on the training split
        preds = pipe.predict(X_test)                       # predict on the test split

        # Compute metrics
        acc = accuracy_score(y_test, preds)                # accuracy on test set
        f1  = f1_score(y_test, preds)                      # F1 on test set

        # Log hyperparameters to MLflow
        mlflow.log_param("C", C)                           # store the C value
        mlflow.log_param("model", "LogisticRegression")    # store the model family

        # Log metrics to MLflow
        mlflow.log_metric("accuracy", acc)                 # store the accuracy
        mlflow.log_metric("f1_score", f1)                  # store the F1

        # Log the trained pipeline as a reusable artifact
        mlflow.sklearn.log_model(pipe, "model")            # saves model files under the run

        print(f"C={C:>5} | acc={acc:.4f} | f1={f1:.4f}")   # progress print

👉 **To browse the runs visually**, open a terminal in this folder and run:

```bash
mlflow ui
```

Then open <http://localhost:5000> in your browser. You will see all four runs side by side with metrics and params.


## 3. Model Registry & Versioning

> *Recall from the slides:* a registry is "a catalog of trained models with versions, stages and metadata — the bridge between training and deployment."

We will pick our **best** run and register the model. In real life you would then promote it: `Staging → Production → Archived`.


In [ ]:
from mlflow.tracking import MlflowClient                  # client to query the tracking server

# Initialize a client pointing at our local store
client = MlflowClient()                                     # connects to the tracking URI

# Find the experiment by name
experiment = client.get_experiment_by_name("breast_cancer_classifier")  # look it up

# Pull all runs from this experiment, ordered by F1 (descending)
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],              # filter on this experiment
    order_by=["metrics.f1_score DESC"],                     # best F1 first
    max_results=5                                           # cap to the top 5
)

# The first run in the list is the best one
best_run = runs[0]                                          # highest-F1 run
print(f"Best run ID: {best_run.info.run_id}")               # show its ID
print(f"Best F1:     {best_run.data.metrics['f1_score']:.4f}")  # show its F1
print(f"Best C:      {best_run.data.params['C']}")          # show its hyperparameter

In [ ]:
# Register the best run's model under a friendly name
model_uri = f"runs:/{best_run.info.run_id}/model"           # URI pointing at the artifact
registered_name = "BreastCancerClassifier"                  # name in the registry

# Register the model — creates version 1, or version N+1 on subsequent runs
result = mlflow.register_model(model_uri, registered_name)  # adds to the registry
print(f"Registered '{registered_name}' as version {result.version}")  # confirmation

## 4. Serving the Model

> *Recall from the slides:* "Wrap as an API — load the model behind a REST/gRPC endpoint (FastAPI, TF Serving, TorchServe)."

We'll save the pipeline to disk, then write a small **FastAPI** app that loads it and exposes a `/predict` endpoint. You can run the app locally with `uvicorn`.


In [ ]:
import joblib                                              # standard serializer for sklearn

# Save the final fitted pipeline to disk
MODEL_PATH = "model.joblib"                                 # file path on disk
joblib.dump(pipeline, MODEL_PATH)                           # serialize the pipeline
print(f"Pipeline saved to {MODEL_PATH} ✓")                  # confirm save

# Load it back to verify the round-trip works
loaded = joblib.load(MODEL_PATH)                            # deserialize
sample = X_test.iloc[[0]]                                   # one example row
print("Reloaded prediction:", loaded.predict(sample)[0])    # should match training-time output

In [ ]:
# Write a small FastAPI app to a .py file
# This is the kind of file you would put in a Docker image and deploy
fastapi_code = '''
# app.py — minimal model server
from fastapi import FastAPI                       # web framework
from pydantic import BaseModel                    # request validation
import joblib                                     # to load the saved pipeline
import numpy as np                                # array handling

app = FastAPI(title="Breast Cancer Classifier")   # create the API

# Load the pipeline ONCE at startup (not on every request)
model = joblib.load("model.joblib")               # the fitted Pipeline

# Define the expected input schema (list of floats)
class Features(BaseModel):                        # pydantic auto-validates types
    values: list[float]                           # one row of feature values

@app.get("/health")                               # health-check endpoint
def health():                                     # used by load balancers
    return {"status": "ok"}                       # return a simple JSON

@app.post("/predict")                             # the main prediction endpoint
def predict(payload: Features):                   # receives the JSON body
    arr = np.array(payload.values).reshape(1, -1) # shape into a 2-D batch of 1
    pred = int(model.predict(arr)[0])             # run the pipeline (scaler + model)
    prob = float(model.predict_proba(arr)[0, 1])  # probability for class 1
    return {"prediction": pred, "probability": prob}  # JSON response
'''

# Write the file
with open("app.py", "w") as f:                              # open for writing
    f.write(fastapi_code)                                   # dump the source
print("Wrote app.py ✓")                                     # confirmation

👉 **To run the server**, open a terminal in this folder:

```bash
uvicorn app:app --reload --port 8000
```

Then in another terminal, hit the endpoint:

```bash
curl -X POST http://localhost:8000/predict \
     -H "Content-Type: application/json" \
     -d '{"values": [14.0, 20.0, 90.0, 600.0, 0.1, 0.2, 0.2, 0.1, 0.2, 0.07, 0.4, 1.0, 3.0, 50.0, 0.005, 0.04, 0.05, 0.02, 0.03, 0.004, 16.0, 25.0, 105.0, 880.0, 0.13, 0.4, 0.5, 0.15, 0.3, 0.08]}'
```

You should get back a JSON like `{"prediction": 0, "probability": 0.02}`.


## 5. Monitoring — Detecting Data Drift

> *Recall from the slides:* "Predictions rarely throw errors — they just get quietly worse. Monitor inputs and outputs, not just uptime."

We will simulate **data drift** by shifting the distribution of one feature, then use a **Kolmogorov-Smirnov test** to detect that the new data no longer matches the training distribution.


In [ ]:
from scipy.stats import ks_2samp                          # two-sample KS test

# Simulate "live" production data by copying X_test and shifting feature 0
X_live = X_test.copy()                                      # start from test data
X_live.iloc[:, 0] = X_live.iloc[:, 0] + 5.0                 # add an offset → drift!

# Pick a feature to monitor
feature_to_check = X_train.columns[0]                       # first feature

# Reference distribution (what we trained on)
ref  = X_train[feature_to_check]                            # training values
# Live distribution (what we're seeing now)
live = X_live[feature_to_check]                             # production-like values

# Run a Kolmogorov-Smirnov test: are the two samples from the same distribution?
stat, p_value = ks_2samp(ref, live)                         # statistic + p-value

print(f"Feature:    {feature_to_check}")                    # show feature name
print(f"KS stat:    {stat:.4f}")                            # KS test statistic
print(f"p-value:    {p_value:.6f}")                         # significance

# Alert if drift detected (p-value very small = significantly different)
if p_value < 0.05:                                          # 5% threshold
    print("⚠️  DRIFT DETECTED — consider retraining the model")
else:
    print("✅ No significant drift")                        # all good

In [ ]:
# Generalize: check every feature and report any that drifted
drift_report = []                                           # collect results here

# Loop over all feature columns
for col in X_train.columns:                                 # one feature at a time
    s, p = ks_2samp(X_train[col], X_live[col])              # KS test per feature
    drift_report.append({"feature": col, "ks": s, "p": p, "drifted": p < 0.05})

# Turn into a DataFrame for easy reading
report_df = pd.DataFrame(drift_report)                      # tabular results
drifted   = report_df[report_df["drifted"]]                 # only the flagged ones

print(f"Total features checked: {len(report_df)}")          # how many we tested
print(f"Drifted features:       {len(drifted)}")            # how many fired
drifted.head()                                              # show the worst offenders

## 🎯 Recap & Homework

**You just built a (small) production ML system:**

| Stage | What you used |
|---|---|
| Pipeline | `sklearn.pipeline.Pipeline` |
| Tracking | `mlflow.log_param/metric/model` |
| Registry | `mlflow.register_model` |
| Serving  | `FastAPI` + `uvicorn` |
| Monitoring | `scipy.stats.ks_2samp` |

### Homework

1. Add a `RandomForestClassifier` run alongside the logistic-regression runs and see which wins on F1.
2. Modify `app.py` to also log every incoming request to a file (foundation for monitoring).
3. Compute KS drift on the **predictions** (output drift) in addition to inputs.
4. Add a `mlflow.log_artifact("model.joblib")` call so the model is saved as a generic artifact too.

➡️ **Next up:** `Lab 2 — Reinforcement Learning Part 1 (MDPs & Dynamic Programming)`.
